In [ ]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, gaussian_dist_time_varying, visualize_gaussian_samples, visualize_multi_task_gaussians, visualize_train_vs_test_gaussians, plot_disturbances_time, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyMLP, rollout_policy, traj_cost, train_nn_policy, meta_train_nn_policy, maml_adapt, evaluate_policy_cost, plot_maml_diagnostics, plot_adapted_losses, plot_adaptation_comparison, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [2]:
# Create and discretize model
A, B, C, D = quad_hover_linear_model()
# print("A =\n", A, "\n\nB =\n", B, "\n\nC =\n", C, "\n\nD =\n", D)

A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]
# print("A_d =\n", A_d, "\n\nB_tilde_d =\n", B_tilde_d, "\n\nC_d =\n", C_d, "\n\nD_d =\n", D_d)

In [48]:
# TRAINING TASKS
point_per_task = 50
task_means  = [-6.0, -4.0, -1.0, 2.0]
task_sigmas = [ 0.2,  0.4,  0.2, 0.3]


tasks_dx = []
tasks_all_disturbances = []
tasks_all_x0 = []

for mean, sigma in zip(task_means, task_sigmas):

    # time-varying disturbance on x
    dx = gaussian_dist_time_varying(mean, sigma, point_per_task)
    tasks_dx.append(dx)

    # nothing on y, gravity on z
    dy = torch.zeros_like(dx)
    dz = -cfg.G * torch.ones_like(dx)

    # combine to get full disturbance for this task, then append to all tasks
    dist = torch.cat([dx, dy, dz], dim=-1)      # shape [50, T, 3]
    tasks_all_disturbances.append(dist)         # shape [num_tasks, 50, T, 3]

    # initial states for all trajectories in this task
    x0 = torch.zeros(point_per_task, 12)
    tasks_all_x0.append(x0)

# just for baseline training
tasks_all_disturbances_joint = torch.cat(tasks_all_disturbances, dim=0)   # [num_tasks * point_per_task, T, 3]
tasks_all_x0_joint = torch.cat(tasks_all_x0, dim=0).float()               # [num_tasks * point_per_task, 12]



# TESTING POINTS
point_per_task = 50
nb_testing_points = 1 * point_per_task

task_target_mean = [6.0]
task_target_sigma = [0.3]
task_target = gaussian_dist_time_varying(task_target_mean[0], task_target_sigma[0], point_per_task, device="cpu")
tasks_target = [task_target]

dist_x_test = torch.cat([task_target], dim=0)                                       # x-axis disturbance (wind)                 
dist_y_test = torch.zeros_like(dist_x_test)                                         # y-axis disturbance (none)                  
dist_z_test = -cfg.G * torch.ones_like(dist_x_test)                                 # z-axis disturbance (gravity)
disturbances_test = torch.cat([dist_x_test, dist_y_test, dist_z_test], dim=-1)                        

# Initial state conditions for testing
x0_test_np = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]*nb_testing_points)
x0_test = torch.from_numpy(x0_test_np).float()



# VISUALIZE TRAINING/TESTING DISTURBANCES
visualize_multi_task_gaussians(tasks_dx, task_means, task_sigmas, "results_only_x/training_dist.png")
visualize_multi_task_gaussians(tasks_target, task_target_mean, task_target_sigma, "results_only_x/testing_dist.png", True)
visualize_train_vs_test_gaussians(tasks_dx, task_means, task_sigmas, tasks_target, task_target_mean, task_target_sigma, "results_only_x/both_training_testing_dist.png")

In [4]:
# JOINT TRAINING (BASELINE)
device = "cpu"
epochs = 5000
compute_mean_every = 100
lr = 1e-3

joint_policy, train_means = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=tasks_all_x0_joint, disturbances=tasks_all_disturbances_joint,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True
)

[ep 100/5000]:   train_loss=6.6186e+04   |    eval_loss=6.0528e+04   |   |x_T|_avg=50.372
[ep 200/5000]:   train_loss=2.6049e+04   |    eval_loss=2.6991e+04   |   |x_T|_avg=40.934
[ep 300/5000]:   train_loss=1.2450e+04   |    eval_loss=1.1602e+04   |   |x_T|_avg=14.661
[ep 400/5000]:   train_loss=3.7245e+03   |    eval_loss=3.5457e+03   |   |x_T|_avg=5.986
[ep 500/5000]:   train_loss=1.2712e+03   |    eval_loss=1.2012e+03   |   |x_T|_avg=4.165
[ep 600/5000]:   train_loss=6.6488e+02   |    eval_loss=6.4443e+02   |   |x_T|_avg=1.666
[ep 700/5000]:   train_loss=4.4745e+02   |    eval_loss=4.4750e+02   |   |x_T|_avg=1.495
[ep 800/5000]:   train_loss=3.3362e+02   |    eval_loss=3.4494e+02   |   |x_T|_avg=1.566
[ep 900/5000]:   train_loss=2.5478e+02   |    eval_loss=2.5137e+02   |   |x_T|_avg=0.606
[ep 1000/5000]:   train_loss=1.9560e+02   |    eval_loss=1.9602e+02   |   |x_T|_avg=0.371
[ep 1100/5000]:   train_loss=1.7021e+02   |    eval_loss=1.6926e+02   |   |x_T|_avg=0.150
[ep 1200/5000]: 

In [ ]:
plot_loss_evolution(train_means, epochs, compute_mean_every, filename="results_only_x/loss_joint_training.png")

In [22]:
# META TRAINING
device = "cpu"
epochs = 5000
compute_mean_every = 100
lr_outer_loop = 1e-3            # β
lr_inner_loop = 1e-3            # α
nb_samples_inner_training = 5
nb_samples_inner_testing = 5


meta_policy, logs = meta_train_nn_policy(
    A_d_nt = A_d, B_tilde_d_nt = B_tilde_d,
    init_points_list = tasks_all_x0,          
    disturbances_list = tasks_all_disturbances,
    lr_outer=lr_outer_loop,          
    lr_inner=lr_inner_loop,   
    meta_batch_size=None, 
    K_inner=nb_samples_inner_training,               
    K_outer=nb_samples_inner_testing,    
    epochs=epochs,
    device="cpu",
    verbose_every=compute_mean_every
)

[MAML] Meta-training on 4 tasks.


/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:501: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  X0 = X0.to(device)


[ep 100/5000] meta_loss = 6.0930e+05
[ep 200/5000] meta_loss = 2.5893e+05
[ep 300/5000] meta_loss = 1.1646e+05
[ep 400/5000] meta_loss = 6.8564e+04
[ep 500/5000] meta_loss = 5.1082e+04
[ep 600/5000] meta_loss = 3.2973e+04
[ep 700/5000] meta_loss = 2.4833e+04
[ep 800/5000] meta_loss = 1.6679e+04
[ep 900/5000] meta_loss = 1.4723e+04
[ep 1000/5000] meta_loss = 1.0870e+04
[ep 1100/5000] meta_loss = 9.6120e+03
[ep 1200/5000] meta_loss = 8.8658e+03
[ep 1300/5000] meta_loss = 3.2150e+03
[ep 1400/5000] meta_loss = 2.1749e+03
[ep 1500/5000] meta_loss = 1.6225e+03
[ep 1600/5000] meta_loss = 1.2379e+03
[ep 1700/5000] meta_loss = 1.0787e+03
[ep 1800/5000] meta_loss = 7.7041e+02
[ep 1900/5000] meta_loss = 9.1378e+02
[ep 2000/5000] meta_loss = 5.4147e+02
[ep 2100/5000] meta_loss = 4.8107e+02
[ep 2200/5000] meta_loss = 4.0338e+02
[ep 2300/5000] meta_loss = 3.9352e+02
[ep 2400/5000] meta_loss = 3.3693e+02
[ep 2500/5000] meta_loss = 3.0676e+02
[ep 2600/5000] meta_loss = 2.8594e+02
[ep 2700/5000] meta_l

In [1]:
plot_maml_diagnostics(logs, filename="results_only_x/loss_meta_training.png")

NameError: name 'plot_maml_diagnostics' is not defined

In [523]:
def plot_adapted_losses(adapt_losses, full_mean_losses, full_min_losses, full_max_losses, filename):
    steps = np.arange(0, len(full_mean_losses))

    plt.figure(figsize=(6,4))
    plt.plot(steps, adapt_losses, marker='o', markersize=4, color="red", alpha=0.8)
    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Inner Loss (5 samples)")
    plt.title("Inner Adaptation Loss (used for weight updates)")
    plt.grid(True)
    plt.xlim(0, 5)
    plt.tight_layout()
    plt.savefig(filename.replace(".png", "_inner.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(6,4))
    plt.plot(steps, full_mean_losses, marker='o', markersize=4, label="Mean Loss", color="red", alpha=0.8)
    plt.fill_between(
        steps,
        full_min_losses,
        full_max_losses,
        alpha=0.15,
        edgecolor="none",
        color="red",
        label="Min-Max Range"
    )

    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.title("Performance on Entire Test Distribution")
    plt.grid(True)
    plt.xlim(0, 5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename.replace(".png", "_full.png"), dpi=300)
    plt.close()


def plot_adaptation_comparison(full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, filename):
    steps = np.arange(0, len(full_mean_losses_joint))
    plt.figure(figsize=(6,4))

    color_meta = "#D81B60"
    plt.plot(steps, full_mean_losses_meta, marker='o', markersize=4, label="Meta adaptation: mean loss", color=color_meta, alpha=0.8)
    plt.fill_between(
        steps,
        full_min_losses_meta,
        full_max_losses_meta,
        alpha=0.15,
        edgecolor="none",
        color=color_meta,
        label="Meta adaptation: min/max range"
    )

    color_joint = "#2E7D32" 
    plt.plot(steps, full_mean_losses_joint, marker='o', markersize=4, label="Joint adaptation: mean loss", color=color_joint, alpha=0.8)
    plt.fill_between(
        steps,
        full_min_losses_joint,
        full_max_losses_joint,
        alpha=0.15,
        edgecolor="none",
        color=color_joint,
        label="Joint adaptation: min/max range"
    )

    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.xlim(0, 5)
    plt.ylim(6e2, 3e4)
    plt.title("Performance on Entire Test Distribution")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

In [241]:
# ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
adapted_policy_from_meta_learning, adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta = maml_adapt(
    meta_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=5e-3,
    nb_gradient_descent=10,
    K_samples=5,
    device="cpu"
)

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:44: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement

[Adapt 1] inner_loss=24403.1289, full_mean=4444.7910
[Adapt 2] inner_loss=8305.4541, full_mean=1614.3988
[Adapt 3] inner_loss=1851.4500, full_mean=1336.0352
[Adapt 4] inner_loss=1365.5647, full_mean=1163.0402
[Adapt 5] inner_loss=1158.9625, full_mean=1052.7975
[Adapt 6] inner_loss=1028.8347, full_mean=1045.2451
[Adapt 7] inner_loss=1021.9792, full_mean=1128.6174
[Adapt 8] inner_loss=1119.2625, full_mean=1043.8729
[Adapt 9] inner_loss=1021.1877, full_mean=1125.0914
[Adapt 10] inner_loss=1116.0530, full_mean=1043.0316


In [456]:
plot_adapted_losses(adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, filename="results_only_x/meta_adaptation.png")

In [ ]:
# APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
adapted_policy_from_joint_learning, adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint = maml_adapt(
    joint_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=5e-3,
    nb_gradient_descent=10,
    K_samples=5,
    device="cpu"
)

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:44: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement

[Adapt 1] inner_loss=1182.6775, full_mean=3441.8818
[Adapt 2] inner_loss=2053.5520, full_mean=5906.9795
[Adapt 3] inner_loss=6442.7930, full_mean=2201.3501
[Adapt 4] inner_loss=2354.4001, full_mean=5895.0542
[Adapt 5] inner_loss=5340.4946, full_mean=2375.7441
[Adapt 6] inner_loss=2339.0886, full_mean=4614.1313
[Adapt 7] inner_loss=5249.2021, full_mean=2739.3816
[Adapt 8] inner_loss=3213.6443, full_mean=3895.6079
[Adapt 9] inner_loss=3276.2168, full_mean=11500.0352
[Adapt 10] inner_loss=11823.4912, full_mean=50004.8047


In [513]:
plot_adapted_losses(adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, filename="results_only_x/joint_adaptation.png")

In [524]:
# COMPARE ADAPTED POLICIES ON TEST TASK
plot_adaptation_comparison(full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, "results_only_x/adaptation_comparison.png")

In [565]:
# Compute improvement percentage plot
meta = np.array(full_mean_losses_meta)
joint = np.array(full_mean_losses_joint)

# Percentage improvement of meta vs joint
improvement_pct = 100 * (joint - meta) / joint

steps = np.arange(0, len(improvement_pct))

plt.figure(figsize=(6,4))
plt.plot(steps, improvement_pct, marker='o', markersize=4, color='purple', alpha=0.8)

plt.xlabel("Adaptation Step")
plt.ylabel("Loss improvement (\%)")
plt.ylim(0, 100)
plt.xlim(0, 5)
plt.title("Improvement of Meta-Learning vs Joint Training")
plt.grid(True)
plt.tight_layout()
plt.savefig("results_only_x/improvement_percentage.png", dpi=300)
plt.close()

In [451]:
# TESTING ADAPTED POLICY FROM META-LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(meta_policy, adapted_policy_from_meta_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_meta, U_test_adapted_from_meta = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device)
)
costs = traj_cost(X_test_adapted_from_meta, U_test_adapted_from_meta)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_meta, filename="results_only_x/testing_xyz_adapted_from_meta.png")
plot_angles_mean(X_test_adapted_from_meta, filename="results_only_x/testing_angles_adapted_from_meta.png")
plot_controls_grid_mean(U_test_adapted_from_meta, filename="results_only_x/testing_controls_adapted_from_meta.png")
animate_trajectories(X_test_adapted_from_meta, "results_only_x/testing_xyz_3d_adapted_from_meta.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/2899939763.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/2899939763.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/2899939763.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  re

Adapted test cost = 1043.0316162109375
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [518]:
# TESTING ADAPTED POLICY FROM JOINT LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(joint_policy, adapted_policy_from_joint_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_joint, U_test_adapted_from_joint = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device)
)
costs = traj_cost(X_test_adapted_from_joint, U_test_adapted_from_joint)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_joint, filename="results_only_x/testing_xyz_adapted_from_joint.png")
plot_angles_mean(X_test_adapted_from_joint, filename="results_only_x/testing_angles_adapted_from_joint.png")
plot_controls_grid_mean(U_test_adapted_from_joint, filename="results_only_x/testing_controls_adapted_from_joint.png")
animate_trajectories(X_test_adapted_from_joint, "results_only_x/testing_xyz_3d_adapted_from_joint.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/3839436553.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/3839436553.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/3839436553.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  re

Adapted test cost = 50004.8046875
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [ ]:
# Robustness: adaptation with different 5 samples from test distribution

all_robust_full_mean_losses_joint = []
all_robust_full_mean_losses_meta = []

for i in range(10):
    print(f"\n=== ROBUSTNESS TEST {i+1} ===")

    # APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
    robust_adapted_policy_from_joint_learning, robust_adapt_losses_joint, robust_full_mean_losses_joint, robust_full_min_losses_joint, robust_full_max_losses_joint = maml_adapt(
        joint_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test,
        lr_inner=5e-3,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_robust_full_mean_losses_joint.append(robust_full_mean_losses_joint)


    # ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
    robust_adapted_policy_from_meta_learning, robust_adapt_losses_meta, robust_full_mean_losses_meta, robust_full_min_losses_meta, robust_full_max_losses_meta = maml_adapt(
        meta_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test,
        lr_inner=5e-3,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_robust_full_mean_losses_meta.append(robust_full_mean_losses_meta)


=== ROBUSTNESS TEST 1 ===
[Adapt 1] inner_loss=5799.2295, full_mean=18164.8906
[Adapt 2] inner_loss=19558.4902, full_mean=5675.1377


/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_86573/307974038.py:44: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement

[Adapt 3] inner_loss=6294.8218, full_mean=4952.8291
[Adapt 4] inner_loss=3108.5864, full_mean=2792.6748
[Adapt 5] inner_loss=2709.6960, full_mean=1464.0830
[Adapt 6] inner_loss=1459.2659, full_mean=1278.6501

=== ROBUSTNESS TEST 2 ===
[Adapt 1] inner_loss=9747.8730, full_mean=3615.0725
[Adapt 2] inner_loss=3259.6023, full_mean=3243.8103
[Adapt 3] inner_loss=3637.9316, full_mean=2249.1611
[Adapt 4] inner_loss=2747.2856, full_mean=1240.2493
[Adapt 5] inner_loss=1379.3308, full_mean=1067.3213
[Adapt 6] inner_loss=1127.6814, full_mean=1114.1965

=== ROBUSTNESS TEST 3 ===
[Adapt 1] inner_loss=11027.0723, full_mean=9888.0000
[Adapt 2] inner_loss=8218.1064, full_mean=4811.1323
[Adapt 3] inner_loss=4602.6895, full_mean=3130.7590
[Adapt 4] inner_loss=2129.1772, full_mean=1625.5288
[Adapt 5] inner_loss=1534.7041, full_mean=1314.1427
[Adapt 6] inner_loss=1343.4338, full_mean=1120.2163

=== ROBUSTNESS TEST 4 ===
[Adapt 1] inner_loss=9786.7637, full_mean=3311.9624
[Adapt 2] inner_loss=2085.1287, fu

In [ ]:
# Plot all robustness curves
plt.figure(figsize=(8, 5))

color_joint = "#2E7D32"   # forest green
color_meta  = "#D81B60"   # magenta

# JOINT TRAINING CURVES (5 curves)
for i, losses in enumerate(all_robust_full_mean_losses_joint):
    plt.plot(
        losses,
        color=color_joint,
        alpha=0.6,
        linewidth=2,
        label="Joint adaptation with 5 samples from test distribution" if i == 0 else None
    )

# META-LEARNING CURVES (5 curves)
for i, losses in enumerate(all_robust_full_mean_losses_meta):
    plt.plot(
        losses,
        color=color_meta,
        alpha=0.6,
        linewidth=2,
        label="Meta adaptation with 5 samples from test distribution" if i == 0 else None
    )

plt.xlabel("Adaptation step")
plt.ylabel("Mean full test loss")
plt.yscale("log")
plt.xlim(0, 5)
plt.grid(True, ls="--", lw=0.4)
plt.legend()
plt.tight_layout()
plt.savefig("results_only_x/robustness_adaptation_comparison.png", dpi=300)
plt.close()